# Notebook de adquisición — Dengue y lluvias 2021-2025

**Pregunta del proyecto:** ¿Qué municipios y semanas epidemiológicas concentran más notificaciones de dengue entre 2021 y 2025, y qué aumentos recurrentes conviene contrastar con las temporadas de lluvia para que las autoridades de salud prioricen la vigilancia, la prevención y el control de criaderos de Aedes?

Este notebook carga las dos fuentes, revisa sus columnas y las prepara a un nivel común de municipio, año y semana antes de cruzarlas. Para ejecutarlo en Google Colab, sube los dos CSV al panel de archivos de la izquierda.


## 1. Importar pandas

In [ ]:
import pandas as pd


## 2. Cargar los archivos CSV

In [ ]:
dengue = pd.read_csv("ins_dengue_municipio_semana_2021_2025.csv")
lluvias = pd.read_csv("lluvias_municipio_2021_2025.csv")


## 3. Revisar dengue

Se observan el número de filas y columnas, los primeros registros, los tipos de datos y los nombres de las columnas.

In [ ]:
print("Dimensiones de dengue:", dengue.shape)
display(dengue.head())
print("Tipos de datos:")
display(dengue.dtypes)
print("Columnas:")
display(dengue.columns)


## 4. Revisar lluvias

Esta fuente contiene observaciones por estación y fecha; todavía no tiene una fila por municipio y semana.

In [ ]:
print("Dimensiones de lluvias:", lluvias.shape)
display(lluvias.head())
print("Tipos de datos:")
display(lluvias.dtypes)
print("Columnas:")
display(lluvias.columns)


## 5. Preparar las columnas de texto

Se crean claves de texto para comparar los nombres de departamento y municipio entre las dos fuentes. Se quitan espacios, se usan mayúsculas y se reemplazan las vocales con tilde. Las columnas originales se conservan.

In [ ]:
tabla_tildes = str.maketrans("ÁÉÍÓÚÜ", "AEIOUU")

for datos in [dengue, lluvias]:
    datos["departamento_clave"] = (
        datos["departamento"].astype(str).str.strip().str.upper().str.translate(tabla_tildes)
    )
    datos["municipio_clave"] = (
        datos["municipio"].astype(str).str.strip().str.upper().str.translate(tabla_tildes)
    )

display(dengue[["departamento", "departamento_clave", "municipio", "municipio_clave"]].head())


## 6. Preparar fecha y valor de lluvias

Los valores que no se puedan convertir se marcarán como faltantes.

In [ ]:
lluvias["fechaobservacion"] = pd.to_datetime(
    lluvias["fechaobservacion"], errors="coerce"
)
lluvias["valorobservado"] = pd.to_numeric(
    lluvias["valorobservado"], errors="coerce"
)


## 7. Filtrar precipitación

Primero se revisan los sensores y las unidades presentes. Se conservan las descripciones que contienen «PRECIPITACION» después de quitar la tilde, incluida la variante «GPRS - PRECIPITACIÓN», y cuya unidad es mm.

In [ ]:
print("Descripciones de sensor:")
display(lluvias["descripcionsensor"].unique())
print("Unidades de medida:")
display(lluvias["unidadmedida"].unique())

lluvias = lluvias[
    lluvias["descripcionsensor"].astype(str).str.upper()
    .str.translate(tabla_tildes).str.contains("PRECIPITACION", na=False)
    & lluvias["unidadmedida"].astype(str).str.strip().str.lower().eq("mm")
].copy()

print("Observaciones de precipitación en mm:", len(lluvias))


## 8. Crear año y semana para lluvias

El dataset de lluvias no contiene semana epidemiológica directamente. Para poder realizar el cruce temporal se obtiene la semana ISO a partir de fechaobservacion. Esta semana funciona como aproximación temporal para la integración y puede presentar diferencias en los límites de año frente al calendario epidemiológico oficial. Se utiliza también el año ISO, que corresponde a esa semana.

In [ ]:
calendario = lluvias["fechaobservacion"].dt.isocalendar()
lluvias["anio"] = calendario.year.astype("Int64")
lluvias["semana"] = calendario.week.astype("Int64")

dengue["anio"] = pd.to_numeric(dengue["anio"], errors="coerce").astype("Int64")
dengue["semana"] = pd.to_numeric(dengue["semana"], errors="coerce").astype("Int64")

filas_antes_filtro = len(lluvias)
lluvias = lluvias.dropna(subset=[
    "codigoestacion", "departamento_clave", "municipio_clave",
    "anio", "semana", "valorobservado"
]).copy()
print("Observaciones sin datos necesarios para agregar:", filas_antes_filtro - len(lluvias))


## 9. Verificar observaciones repetidas

Es normal encontrar varias observaciones de una estación en el mismo municipio y semana: pueden corresponder a diferentes días y horas. Aquí se cuentan antes de acumularlas.

In [ ]:
columnas_estacion = [
    "codigoestacion", "departamento_clave", "municipio_clave", "anio", "semana"
]
conteo_observaciones = (
    lluvias.groupby(columnas_estacion, as_index=False).size()
    .rename(columns={"size": "numero_observaciones"})
)
repetidas = conteo_observaciones[conteo_observaciones["numero_observaciones"] > 1]
print("Combinaciones estación-municipio-año-semana con más de una observación:", len(repetidas))
display(repetidas.head())


## 10. Agregar lluvia por estación y semana

Primero se acumulan las observaciones de precipitación de cada estación durante una misma semana.

In [ ]:
lluvia_estacion_semana = (
    lluvias.groupby(columnas_estacion, as_index=False)["valorobservado"].sum()
)
lluvia_estacion_semana = lluvia_estacion_semana.rename(
    columns={"valorobservado": "precipitacion_estacion_semana_mm"}
)
display(lluvia_estacion_semana.head())


## 11. Agregar lluvia por municipio y semana

Cuando existen varias estaciones dentro de un municipio, se utiliza el promedio de la precipitación semanal registrada por las estaciones para obtener un único valor municipal y evitar que los municipios con más estaciones acumulen lluvia de manera artificial.

In [ ]:
llave_cruce = ["departamento_clave", "municipio_clave", "anio", "semana"]
lluvias_semana = (
    lluvia_estacion_semana.groupby(llave_cruce, as_index=False).agg(
        precipitacion_mm=("precipitacion_estacion_semana_mm", "mean"),
        numero_estaciones=("codigoestacion", "nunique")
    )
)


## 12. Revisar las lluvias agregadas

La llave municipal y semanal debe aparecer una sola vez en `lluvias_semana`.

In [ ]:
print("Dimensiones de lluvias agregadas:", lluvias_semana.shape)
display(lluvias_semana.head())
print("Filas duplicadas por la llave del cruce:", lluvias_semana.duplicated(subset=llave_cruce).sum())


## 13. Verificar dengue

Se comprueban duplicados con la misma llave del cruce. Si aparecen, se suman sus notificaciones sin descartar filas silenciosamente.

In [ ]:
duplicados_dengue = dengue.duplicated(subset=llave_cruce).sum()
print("Filas duplicadas de dengue por departamento, municipio, año y semana:", duplicados_dengue)

if duplicados_dengue == 0:
    dengue_semana = dengue.copy()
else:
    dengue_semana = dengue.groupby(
        ["anio", "semana", "codigo_municipio", "departamento", "municipio",
         "departamento_clave", "municipio_clave"],
        as_index=False
    )["casos_ins"].sum()
    print("Filas de dengue después de sumar duplicados:", len(dengue_semana))


## 14. Llave del cruce

Las dos fuentes se encuentran ahora al mismo nivel territorial y temporal. El cruce se realizará mediante departamento, municipio, año y semana. La llave es `departamento_clave`, `municipio_clave`, `anio` y `semana`.

## 15. Filas antes del cruce

In [ ]:
print("Filas de dengue:", len(dengue_semana))
print("Filas de lluvias:", len(lluvias_semana))


## 16. Cruzar las fuentes

El cruce izquierdo conserva los registros de dengue, aun cuando no encuentren información de lluvia.

In [ ]:
datos_cruzados = pd.merge(
    dengue_semana,
    lluvias_semana,
    on=["departamento_clave", "municipio_clave", "anio", "semana"],
    how="left"
)
datos_cruzados["numero_estaciones"] = datos_cruzados["numero_estaciones"].astype("Int64")


## 17. Verificar las filas después del cruce

In [ ]:
print("Filas antes del cruce:", len(dengue_semana))
print("Filas después del cruce:", len(datos_cruzados))

if len(dengue_semana) == len(datos_cruzados):
    print("El cruce conservó el número de filas del dataset de dengue.")
else:
    print("ADVERTENCIA: revisar el cruce porque cambió el número de filas.")

print("Dimensiones del resultado:", datos_cruzados.shape)
display(datos_cruzados.head())


## 18. Registros sin lluvia

Estos registros corresponden a combinaciones de municipio, año y semana presentes en el dataset de dengue que no encontraron una observación equivalente en la fuente de precipitaciones.

In [ ]:
sin_lluvia = datos_cruzados[
    datos_cruzados["precipitacion_mm"].isna()
]
print("Registros sin información de lluvia:", len(sin_lluvia))
display(sin_lluvia.head())


## 19. Porcentaje de cobertura

Se calcula qué proporción de registros de dengue encontró información de lluvia.

In [ ]:
total = len(datos_cruzados)
con_lluvia = datos_cruzados["precipitacion_mm"].notna().sum()
porcentaje = con_lluvia / total * 100
print("Porcentaje de registros de dengue con información de lluvia:", round(porcentaje, 2), "%")


**Interpretación:** un valor faltante en `precipitacion_mm` indica que no se encontró una observación equivalente en la fuente de lluvias; no significa que la lluvia haya sido cero. La cobertura debe revisarse antes de comparar ambas fuentes.

## 20. Verificar los años

In [ ]:
print("Años presentes en el resultado:", sorted(datos_cruzados["anio"].dropna().unique()))


## 21. Diccionario de datos

Este diccionario describe las columnas principales del archivo cruzado.

In [ ]:
diccionario_datos = pd.DataFrame(
    [
        ["anio", "entero", "Ambas fuentes", "Año del registro de dengue y año ISO usado para agrupar la lluvia."],
        ["semana", "entero", "Ambas fuentes", "Semana epidemiológica del dengue y semana ISO utilizada como aproximación temporal para el cruce."],
        ["codigo_municipio", "entero", "ins_dengue_municipio_semana_2021_2025.csv", "Código del municipio disponible en la fuente del INS."],
        ["departamento", "texto", "ins_dengue_municipio_semana_2021_2025.csv", "Nombre original del departamento en el registro de dengue."],
        ["municipio", "texto", "ins_dengue_municipio_semana_2021_2025.csv", "Nombre original del municipio en el registro de dengue."],
        ["casos_ins", "entero", "ins_dengue_municipio_semana_2021_2025.csv", "Número de notificaciones de dengue registradas por el INS para el municipio y semana."],
        ["departamento_clave", "texto", "Ambas fuentes", "Departamento normalizado para realizar el cruce."],
        ["municipio_clave", "texto", "Ambas fuentes", "Municipio normalizado para realizar el cruce."],
        ["precipitacion_mm", "decimal", "lluvias_municipio_2021_2025.csv", "Promedio municipal de la precipitación semanal registrada por las estaciones meteorológicas disponibles, en mm."],
        ["numero_estaciones", "entero", "lluvias_municipio_2021_2025.csv", "Número de estaciones utilizadas para calcular la precipitación del municipio durante esa semana."],
    ],
    columns=["Columna", "Tipo de dato", "Fuente", "Significado"]
)
display(diccionario_datos)


## 22. Guardar el resultado

El CSV se genera en la carpeta de trabajo de Colab y puede descargarse desde el panel de archivos.

In [ ]:
datos_cruzados.to_csv(
    "dengue_lluvias_municipio_semana_2021_2025.csv",
    index=False
)
print("Archivo generado correctamente.")


## Conclusión

Se cargaron las fuentes de dengue y precipitación correspondientes al periodo 2021-2025. Debido a que la fuente de precipitaciones contiene múltiples observaciones por estación y fecha, se agregó primero la información a nivel semanal y posteriormente a nivel municipal. Después se realizó un cruce izquierdo utilizando departamento, municipio, año y semana, tomando los registros de dengue como fuente principal. Finalmente se verificó el número de filas y se identificaron los registros que no encontraron información de precipitación. La semana ISO de lluvias es una aproximación temporal y el cruce no permite afirmar causalidad entre lluvia y dengue.